# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. ML Task Framing: Refresh & Content Opportunity Scoring

I am framing the Content Refresh lane as a supervised probabilistic scoring task. For each page, the model estimates a priority score $P(\text{Decline} \mid \text{Features})$. That score is then used to rank the queue of pages presented to editors, making the model a decision-support tool rather than an automated publishing decision.

> The system is framed as a pointwise scoring problem during training and a listwise ranking problem at inference time, since the business action is to prioritize a limited review queue.

In [14]:
import pandas as pd
from pathlib import Path
from IPython.display import display

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    data_path = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
    if data_path.exists():
        break
    data_path = None

if data_path is None:
    raise FileNotFoundError("Could not find the starter dataset from the current notebook location.")

# Load the starter slice for the refresh lane.
df = pd.read_csv(data_path)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
print("Declining share:", round(df["is_declining_label"].mean(), 3))
print(df["is_declining_label"].value_counts().to_dict())

Rows: 30000 | Columns: 45
Declining share: 0.542
{1: 16262, 0: 13738}


## 2. Target & Proxy Variable Formulation

The target variable is a binary proxy: `is_declining_label` ($1$ if the recent 30-day impression trend is labeled as "down", $0$ otherwise). This is an observed statistical outcome in the data rather than a manually written business rule, although it still serves as an empirical proxy for a page needing a content refresh.

> To prevent target leakage during model training, target-defining columns such as `trend_direction` and `trend_pct` will be excluded from the feature matrix so the model learns from indirect signals like traffic decay, content age, content type, and length.

In [10]:
# Preview the proxy target alongside the raw trend fields.
target_preview = df[["content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"]].head(10)
display(target_preview)

,content_id,client_id,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,down,-38.9,1
6,content_9a34b442b552,client_8722616204,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,stable,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,down,-58.8,1
9,content_c27558df2b0c,client_19581e27de,down,-29.2,1


## 3. Evaluation Metric & Business Impact Alignment

The primary evaluation metric for this model is Precision@50. If an editorial team can review roughly 50 pages per week, Precision@50 measures the share of those manual reviews that were spent on pages genuinely needing updates, which directly reduces wasted effort and improves the return on human review time.

> A secondary metric such as ROC-AUC or NDCG@50 would be useful for assessing ranking quality across the broader top tier of candidates, but Precision@50 matches the weekly human-capacity constraint most closely.

In [11]:
# A simple sanity check for the target distribution before any modeling.
positive_rate = df["is_declining_label"].mean()
print(f"Base-rate of declining proxy: {positive_rate:.3f}")
print("A model that beats this rate on Precision@50 would be useful for review triage.")

Base-rate of declining proxy: 0.542
A model that beats this rate on Precision@50 would be useful for review triage.


## 4. The Unit of Analysis, as a Real Dataframe

In this starter slice, one row is one content item (one page/article). The dataframe therefore represents a set of candidate pages that could be reviewed for refresh, with each row carrying content metadata, traffic history, and the observed decline proxy.

> This makes the unit of analysis explicit: the model is not predicting at the client level or the keyword level; it is scoring individual content pages for editorial triage.

In [12]:
# Show the unit of analysis directly.
# Features for training will exclude 'trend_direction' and 'trend_pct' to avoid label leakage.
unit_of_analysis = df[["content_id", "client_id", "content_type", "word_count", "impressions_90d", "trend_direction", "is_declining_label"]].head(8)
display(unit_of_analysis)

,content_id,client_id,content_type,word_count,impressions_90d,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,3803,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,15320,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,12581,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,11751,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,19140,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3080.0,3970,down,1
6,content_9a34b442b552,client_8722616204,keyword article,3059.0,20,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,NaN,1724,stable,0


## 5. Why Machine Learning Beats Static Heuristics

A static business rule such as `IF impressions_90d < 1000 THEN review` breaks down because content decay is driven by non-linear interactions among traffic volume, recent trend, freshness, content type, search intent, position, and client-specific behavior. Hand-authored thresholds are brittle and cannot adapt well to different content categories or to changing search-engine dynamics over time.

> Fixed heuristics fail because SEO decline is a high-dimensional, noisy, and shifting pattern. Machine learning can model complex feature interactions and be retrained as traffic dynamics evolve, making it more robust than static if-then rules.

In [13]:
# A final quick check: the action is to rank a review queue, not to make a deterministic rule.
print("This framing supports an editor action: review the highest-scoring pages first.")
print("The model output is a score for triage, and the label is an observed proxy for decline.")

This framing supports an editor action: review the highest-scoring pages first.
The model output is a score for triage, and the label is an observed proxy for decline.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.